# 03 generate 到底做了什么：数字到回答

今天只解决一个问题：

> 模型拿到 input_ids 以后，怎么变成一段中文回答？

核心入口是 `model.generate(...)`。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("本 micro notebook 默认只读代码；真实模型推理开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=80):
    lines = read_text(rel).splitlines()
    print(f"--- {rel}:{start}-{min(end, len(lines))} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = [i for i, line in enumerate(lines, start=1) if keyword in line]
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        show_file(rel, max(1, hit-context), min(len(lines), hit+context))
        print()

## 1. generate 在代码里的位置

In [ ]:
find_lines("scripts/infer.py", "def generate", context=8)
find_lines("scripts/infer.py", "model.generate", context=12)

## 2. generation_kwargs 是什么？

`generation_kwargs` 就是传给模型生成函数的一包参数：`input_ids`、`max_new_tokens`、`temperature`、`top_p`、`eos_token_id` 等。

In [ ]:
find_lines("scripts/infer.py", "generation_kwargs", context=14)

## 3. 为什么 temperature=0 适合评估？

项目做固定 prompt 对比时，希望同一个模型每次回答尽量一致。`temperature=0` 会关闭采样，更适合比较 base、public-SFT、custom-SFT、DPO。

## 4. 为什么只 decode 新 token？

模型输出通常包含 `原 prompt token + 新生成 token`。如果全部 decode，会把用户问题也打印出来。项目只取 `prompt_len:` 之后的新 token。

In [ ]:
find_lines("scripts/infer.py", "prompt_len", context=10)
find_lines("scripts/infer.py", "new_tokens", context=8)

## 5. adapter 如何改变 generate 的结果？

generate 入口还是同一个函数。LoRA adapter 改变的是模型内部参数路径，所以同一个 prompt 下输出会变。

In [ ]:
find_lines("scripts/infer.py", "PeftModel.from_pretrained", context=8)

## 6. 安全练习：比较命令，不默认执行

In [ ]:
prompt = "SFT 是什么？它和 LoRA 是什么关系？"
base_cmd = [sys.executable, "scripts/infer.py", "--prompt", prompt, "--max_new_tokens", "120", "--temperature", "0", "--local_files_only"]
sft_cmd = base_cmd + ["--adapter_path", "outputs/sft_lora_qwen05b_custom_v3_from_v1_patch"]
print("base 命令:")
print(" ".join(base_cmd))
print("\nSFT adapter 命令:")
print(" ".join(sft_cmd))
RUN_COMPARE = False
if RUN_COMPARE:
    for name, cmd in [("base", base_cmd), ("sft", sft_cmd)]:
        print("\n===", name, "===")
        result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True, encoding="utf-8", errors="replace")
        print(result.stdout)
else:
    print("默认不运行。这里只先看命令结构。")

## 7. 本节面试三句话

1. `model.generate` 接收 prompt token，并自回归地产生新 token。
2. 项目只 decode prompt 后面的新 token，所以输出是纯回答。
3. LoRA/SFT/DPO 不改变 generate 入口，而是改变模型内部参数，从而改变同一个 prompt 的回答。